----
<div style="display: flex; align-items: center;">
    <img src="https://developers.lseg.com/content/dam/devportal/icons/logo/lseg-logo.svg" width="20%" style="vertical-align: top;">
    <h1 style="margin-left: 20px;">Data Library for Python</h1>
</div>

----

## Content layer - Filings - Search
This notebook demonstrates how to search financial filings using `lseg.data.content.filings.search`: organization and form filters, filing dates, automatic pagination, a custom GraphQL query and asynchronous requests.

**Prerequisites:** a configured platform session entitled to the Filings services. Configure your credentials in `lseg-data.config.json`, not in the notebook. Examples use organization ID, historical date ranges and query ; results depend on current service availability and entitlements. Service and authentication errors are not suppressed.

Searching returns document metadata and downloadable file objects; it does not download document contents. See [Retrieval and Downloads](EX-2.12.02-Filings-RetrievalAndDownloads.ipynb) for the next step. Both notebooks can be run independently.

#### Learn more

Visit the [LSEG Developer Portal](https://developers.lseg.com/en) for learning materials about the LSEG Data Library for Python.

#### Getting Help and Support

For questions about the API, visit the [LSEG Developer Community](https://community.developers.lseg.com/).

----

## Set the configuration file location
Run the notebook from its own directory. As in the other Content layer examples, `LD_LIB_CONFIG_PATH` points to the project's `Configuration` directory.

In [ ]:
import os
os.environ["LD_LIB_CONFIG_PATH"] = "../../../Configuration"

## Some Imports to start with

In [ ]:
import asyncio
from datetime import date, datetime

import lseg.data as ld
from lseg.data.content import filings
from lseg.data.content.filings import Feed

## Open the data session
Select the `platform.rdp` session configured in `lseg-data.config.json`. Change the session name if your entitled platform session has a different name.

In [ ]:
ld.open_session("platform.rdp")

## Search by organization
`org_id` is an organization identifier, not a RIC. The tests use `4295908552`. `limit` controls the requested number of filing hits and defaults to 10 when omitted. `sort_order` defaults to `DESC` (newest filing dates first).

The search DataFrame contains `DocumentTitle`, `Filename`, `MimeType`, `Dcn`, `DocId` and `FinancialFilingId`. A filing can have several file formats, so the number of DataFrame rows or downloadable files can exceed the number of filing hits.

In [ ]:
organization_response = filings.search.Definition(
    org_id="4295908552",
    start_date=date(2022, 3, 1),
    limit=5,
).get_data()
organization_response.data.df

## Search by feed, form type and filing dates
Use `Feed.EDGAR` or the equivalent string `"Edgar"`. `form_type="10-Q"` restricts results to quarterly filings. Dates can be supplied as Python `date` or `datetime` objects or date strings. The following request combines the date representations exercised in the tests.

In [ ]:
response = filings.search.Definition(
    form_type="10-Q",
    feed=Feed.EDGAR,
    start_date=datetime(2022, 3, 1, 14, 30),
    end_date="2023-03-01",
    sort_order="DESC",
    limit=3,
).get_data()
response.data.df

### Inspect downloadable file metadata
`response.data.files` contains file objects with `filename`, `title`, `mimetype`, `dcn`, `doc_id` and `filing_id` attributes. They expose `download()` and `download_async()` methods. This cell only inspects metadata; it does not download anything.

In [ ]:
for file in response.data.files:
    print(file.filename, file.mimetype, file.doc_id)

## Automatic pagination
For parameter-based searches, the library requests up to 200 filing hits per service call and follows cursors until the requested `limit` is reached or the service has no more hits. The tests request 220 filings for February 2023 to exercise a second page. Counts shown below reflect the actual response, not an assumption that enough results always exist.

`data.raw` can be a dictionary for one response or a list for a multi-page response. The DataFrame combines file metadata across pages. Additional fields, such as `FilingDate` and `_metadata.totalCount`, remain available in the raw payload.

In [ ]:
paged_response = filings.search.Definition(
    form_type="10-Q",
    feed="Edgar",
    sort_order="ASC",
    start_date="2023-02-01",
    end_date="2023-02-28",
    limit=220,
).get_data()

raw = paged_response.data.raw
pages = raw if isinstance(raw, list) else [raw]
for page_number, page in enumerate(pages, start=1):
    hits = page["data"]["FinancialFiling"]
    print(f"Page {page_number}: {len(hits)} filing hits")

display(paged_response.data.df.head())

## Submit a custom GraphQL query
For more control over filters and selected fields, provide a complete query as in the tests. Build and validate queries against the Filings data-store schema in the API Playground.

**Do not mix modes:** `query` and optional `variables` cannot be combined with convenience parameters such as `form_type`, `feed`, `org_id`, dates, `limit` or `sort_order`. Put these constraints inside the GraphQL query instead. Custom queries are sent once; the library does not automatically paginate them.

In [ ]:
query = """
{
  FinancialFiling(
    filter: {AND: [
      {FilingDocument: {DocumentSummary: {FeedName: {EQ: "Edgar"}}}},
      {FilingDocument: {DocumentSummary: {FormType: {EQ: "10-Q"}}}},
      {FilingDocument: {DocumentSummary: {FilingDate: {BETWN: {
        FROM: "2020-10-01T00:00:00Z", TO: "2020-12-31T00:00:00Z"
      }}}}}
    ]}
    sort: {FilingDocument: {DocumentSummary: {FilingDate: DESC}}}
    limit: 3
  ) {
    _metadata { totalCount }
    FilingDocument {
      Identifiers { Dcn }
      DocId
      FinancialFilingId
      DocumentSummary {
        DocumentTitle FeedName FormType HighLevelCategory MidLevelCategory
        FilingDate SecAccessionNumber SizeInBytes
      }
      FilesMetaData { FileName MimeType }
    }
  }
}
"""

query_response = filings.search.Definition(query=query).get_data()
query_response.data.df

In [ ]:
query_response.data.raw

## Run independent searches asynchronously
`get_data_async()` is the asynchronous counterpart of `get_data()` and also supports automatic pagination for parameter-based searches. Use `asyncio.gather` for independent requests. Jupyter supports top-level `await`; do not call `asyncio.run()` inside the notebook event loop. Errors propagate to the cell.

In [ ]:
ascending_response, descending_response = await asyncio.gather(
    filings.search.Definition(
        form_type="10-Q",
        feed=Feed.EDGAR,
        start_date="2023-02-01",
        end_date="2023-02-28",
        sort_order="ASC",
        limit=3,
    ).get_data_async(),
    filings.search.Definition(
        form_type="10-Q",
        feed=Feed.EDGAR,
        start_date="2023-02-01",
        end_date="2023-02-28",
        sort_order="DESC",
        limit=3,
    ).get_data_async(),
)
display(ascending_response.data.df)
display(descending_response.data.df)

## Close the session
No documents were downloaded and no server-side resources were created. If a request fails, run this cell to close the session.

In [ ]:
ld.close_session()